# Neural Architecture Search and AutoML Framework

This notebook implements a comprehensive neural architecture search (NAS) system with:
- Automated architecture optimization using Optuna
- Hyperparameter tuning
- Multiple search strategies (Random, Bayesian, Evolutionary)
- Performance tracking and visualization
- Model interpretation

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR, ReduceLROnPlateau
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

# AutoML and NAS
import optuna
from optuna.pruners import MedianPruner, HyperbandPruner
from optuna.samplers import TPESampler, RandomSampler, CmaEsSampler
from optuna.visualization import plot_optimization_history, plot_param_importances

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Utilities
from typing import Dict, List, Optional, Tuple, Any, Union
from dataclasses import dataclass, field
import warnings
import time
import pickle
import json
from pathlib import Path

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

## 1. Configuration and Data Classes

In [ ]:
@dataclass
class NASConfig:
    """Configuration for Neural Architecture Search."""
    # Search space
    min_layers: int = 1
    max_layers: int = 10
    min_units: int = 16
    max_units: int = 1024
    activations: List[str] = field(default_factory=lambda: ['relu', 'elu', 'leaky_relu', 'selu', 'gelu'])
    dropout_min: float = 0.0
    dropout_max: float = 0.5
    
    # Training
    max_epochs: int = 100
    patience: int = 10
    batch_sizes: List[int] = field(default_factory=lambda: [16, 32, 64, 128, 256])
    learning_rates: Tuple[float, float] = (1e-5, 1e-1)
    optimizers: List[str] = field(default_factory=lambda: ['adam', 'adamw', 'sgd', 'rmsprop'])
    
    # Search
    n_trials: int = 100
    timeout: int = 3600  # seconds
    n_jobs: int = 1
    pruner_type: str = 'median'
    sampler_type: str = 'tpe'
    

@dataclass
class SearchResult:
    """Results from architecture search."""
    best_params: Dict[str, Any]
    best_value: float
    best_model: nn.Module
    study: optuna.Study
    training_history: Dict[str, List[float]]
    search_time: float
    n_completed_trials: int
    n_pruned_trials: int

## 2. Neural Network Building Blocks

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with skip connection."""
    
    def __init__(self, in_features: int, out_features: int, 
                 activation: str = 'relu', dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(in_features, out_features)
        self.fc2 = nn.Linear(out_features, out_features)
        self.activation = self._get_activation(activation)
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.BatchNorm1d(out_features)
        self.norm2 = nn.BatchNorm1d(out_features)
        
        # Skip connection
        self.skip = nn.Linear(in_features, out_features) if in_features != out_features else None
    
    def _get_activation(self, name: str) -> nn.Module:
        activations = {
            'relu': nn.ReLU(),
            'elu': nn.ELU(),
            'leaky_relu': nn.LeakyReLU(0.1),
            'selu': nn.SELU(),
            'gelu': nn.GELU(),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid()
        }
        return activations.get(name, nn.ReLU())
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x if self.skip is None else self.skip(x)
        
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = self.norm2(x)
        
        x = x + residual
        x = self.activation(x)
        
        return x


class AttentionLayer(nn.Module):
    """Self-attention layer."""
    
    def __init__(self, in_features: int, n_heads: int = 8):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = in_features // n_heads
        
        self.query = nn.Linear(in_features, in_features)
        self.key = nn.Linear(in_features, in_features)
        self.value = nn.Linear(in_features, in_features)
        self.out = nn.Linear(in_features, in_features)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = x.shape[0], x.shape[1] if len(x.shape) > 2 else 1
        
        # Reshape for multi-head attention
        if len(x.shape) == 2:
            x = x.unsqueeze(1)  # Add sequence dimension
        
        Q = self.query(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.key(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.value(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.head_dim)
        attention = F.softmax(scores, dim=-1)
        context = torch.matmul(attention, V)
        
        # Reshape and project
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        output = self.out(context)
        
        if seq_len == 1:
            output = output.squeeze(1)  # Remove sequence dimension if it was added
        
        return output


class DynamicNetwork(nn.Module):
    """Dynamic neural network built from configuration."""
    
    def __init__(self, config: Dict[str, Any], input_dim: int, output_dim: int):
        super().__init__()
        self.config = config
        self.layers = nn.ModuleList()
        
        # Build network from config
        prev_dim = input_dim
        
        for i in range(config.get('n_layers', 1)):
            layer_type = config.get(f'layer_type_l{i}', 'linear')
            n_units = config.get(f'n_units_l{i}', 128)
            
            if layer_type == 'linear':
                self.layers.append(nn.Linear(prev_dim, n_units))
                self.layers.append(nn.BatchNorm1d(n_units))
                self.layers.append(self._get_activation(config.get(f'activation_l{i}', 'relu')))
                self.layers.append(nn.Dropout(config.get(f'dropout_l{i}', 0.1)))
                prev_dim = n_units
                
            elif layer_type == 'residual':
                self.layers.append(ResidualBlock(
                    prev_dim, n_units,
                    activation=config.get(f'activation_l{i}', 'relu'),
                    dropout=config.get(f'dropout_l{i}', 0.1)
                ))
                prev_dim = n_units
                
            elif layer_type == 'attention':
                self.layers.append(AttentionLayer(prev_dim, n_heads=config.get(f'n_heads_l{i}', 8)))
        
        # Output layer
        self.output_layer = nn.Linear(prev_dim, output_dim)
    
    def _get_activation(self, name: str) -> nn.Module:
        activations = {
            'relu': nn.ReLU(),
            'elu': nn.ELU(),
            'leaky_relu': nn.LeakyReLU(0.1),
            'selu': nn.SELU(),
            'gelu': nn.GELU(),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid(),
            'swish': nn.SiLU()
        }
        return activations.get(name, nn.ReLU())
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return self.output_layer(x)

## 3. Neural Architecture Search Implementation

In [ ]:
class NeuralArchitectureSearch:
    """Automated neural architecture search with Optuna."""
    
    def __init__(self, 
                 input_dim: int,
                 output_dim: int,
                 task_type: str = 'classification',
                 config: NASConfig = None,
                 device: str = None):
        """
        Initialize NAS.
        
        Parameters:
        -----------
        input_dim : int
            Input dimension
        output_dim : int
            Output dimension
        task_type : str
            Task type ('classification', 'regression', 'multi_label')
        config : NASConfig
            Search configuration
        device : str
            Device to use ('cuda', 'cpu', or None for auto)
        """
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.task_type = task_type
        self.config = config or NASConfig()
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        
        self.best_model = None
        self.study = None
        self.search_history = []
        
    def create_model(self, trial: optuna.Trial) -> nn.Module:
        """Create a model based on trial suggestions."""
        config = {}
        
        # Number of layers
        n_layers = trial.suggest_int('n_layers', self.config.min_layers, self.config.max_layers)
        config['n_layers'] = n_layers
        
        # Configure each layer
        for i in range(n_layers):
            # Layer type
            layer_type = trial.suggest_categorical(f'layer_type_l{i}', ['linear', 'residual', 'attention'])
            config[f'layer_type_l{i}'] = layer_type
            
            # Number of units
            n_units = trial.suggest_int(f'n_units_l{i}', 
                                       self.config.min_units, 
                                       self.config.max_units,
                                       log=True)
            config[f'n_units_l{i}'] = n_units
            
            # Activation
            activation = trial.suggest_categorical(f'activation_l{i}', self.config.activations)
            config[f'activation_l{i}'] = activation
            
            # Dropout
            dropout = trial.suggest_float(f'dropout_l{i}', 
                                        self.config.dropout_min,
                                        self.config.dropout_max)
            config[f'dropout_l{i}'] = dropout
            
            # Additional params for specific layer types
            if layer_type == 'attention':
                n_heads = trial.suggest_categorical(f'n_heads_l{i}', [1, 2, 4, 8])
                config[f'n_heads_l{i}'] = n_heads
        
        # Create model
        model = DynamicNetwork(config, self.input_dim, self.output_dim)
        
        return model.to(self.device)
    
    def objective(self, trial: optuna.Trial, 
                 X_train: np.ndarray, y_train: np.ndarray,
                 X_val: np.ndarray, y_val: np.ndarray) -> float:
        """Objective function for Optuna optimization."""
        
        # Create model
        model = self.create_model(trial)
        
        # Training hyperparameters
        lr = trial.suggest_loguniform('learning_rate', 
                                     self.config.learning_rates[0],
                                     self.config.learning_rates[1])
        batch_size = trial.suggest_categorical('batch_size', self.config.batch_sizes)
        optimizer_name = trial.suggest_categorical('optimizer', self.config.optimizers)
        
        # Additional training params
        weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-2)
        
        # Train and evaluate
        score = self._train_and_evaluate(
            model, X_train, y_train, X_val, y_val,
            lr, batch_size, optimizer_name, weight_decay, trial
        )
        
        return score
    
    def _train_and_evaluate(self, model: nn.Module,
                           X_train: np.ndarray, y_train: np.ndarray,
                           X_val: np.ndarray, y_val: np.ndarray,
                           lr: float, batch_size: int, 
                           optimizer_name: str, weight_decay: float,
                           trial: optuna.Trial = None) -> float:
        """Train and evaluate a model."""
        
        # Prepare data
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train).to(self.device),
            torch.LongTensor(y_train).to(self.device) if self.task_type == 'classification' 
            else torch.FloatTensor(y_train).to(self.device)
        )
        val_dataset = TensorDataset(
            torch.FloatTensor(X_val).to(self.device),
            torch.LongTensor(y_val).to(self.device) if self.task_type == 'classification'
            else torch.FloatTensor(y_val).to(self.device)
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Loss function
        if self.task_type == 'classification':
            criterion = nn.CrossEntropyLoss()
        elif self.task_type == 'multi_label':
            criterion = nn.BCEWithLogitsLoss()
        else:
            criterion = nn.MSELoss()
        
        # Optimizer
        if optimizer_name == 'adam':
            optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        elif optimizer_name == 'adamw':
            optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        elif optimizer_name == 'sgd':
            optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
        else:
            optimizer = optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        # Learning rate scheduler
        scheduler = CosineAnnealingLR(optimizer, T_max=self.config.max_epochs, eta_min=1e-6)
        
        # Training loop
        best_val_score = float('-inf') if self.task_type == 'classification' else float('inf')
        patience_counter = 0
        
        for epoch in range(self.config.max_epochs):
            # Training
            model.train()
            train_loss = 0
            
            for batch_idx, (data, target) in enumerate(train_loader):
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                train_loss += loss.item()
            
            # Validation
            model.eval()
            val_loss = 0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for data, target in val_loader:
                    output = model(data)
                    val_loss += criterion(output, target).item()
                    
                    if self.task_type == 'classification':
                        pred = output.argmax(dim=1)
                        val_correct += pred.eq(target).sum().item()
                        val_total += target.size(0)
            
            # Calculate validation score
            if self.task_type == 'classification':
                val_score = val_correct / val_total
            else:
                val_score = -val_loss / len(val_loader)  # Negative for minimization
            
            # Early stopping
            if self.task_type == 'classification':
                if val_score > best_val_score:
                    best_val_score = val_score
                    patience_counter = 0
                else:
                    patience_counter += 1
            else:
                if val_score < best_val_score:
                    best_val_score = val_score
                    patience_counter = 0
                else:
                    patience_counter += 1
            
            if patience_counter >= self.config.patience:
                break
            
            # Update scheduler
            scheduler.step()
            
            # Report to Optuna
            if trial is not None:
                trial.report(val_score, epoch)
                
                # Pruning
                if trial.should_prune():
                    raise optuna.TrialPruned()
        
        return best_val_score
    
    def search(self, X_train: np.ndarray, y_train: np.ndarray,
              X_val: np.ndarray, y_val: np.ndarray,
              study_name: str = 'NAS_Study') -> SearchResult:
        """Perform neural architecture search."""
        
        print(f"Starting Neural Architecture Search...")
        print(f"Search space: {self.config.min_layers}-{self.config.max_layers} layers, "
              f"{self.config.min_units}-{self.config.max_units} units")
        print(f"Number of trials: {self.config.n_trials}")
        print(f"Device: {self.device}\n")
        
        start_time = time.time()
        
        # Create sampler
        if self.config.sampler_type == 'tpe':
            sampler = TPESampler(seed=42)
        elif self.config.sampler_type == 'random':
            sampler = RandomSampler(seed=42)
        elif self.config.sampler_type == 'cmaes':
            sampler = CmaEsSampler(seed=42)
        else:
            sampler = TPESampler(seed=42)
        
        # Create pruner
        if self.config.pruner_type == 'median':
            pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=10)
        elif self.config.pruner_type == 'hyperband':
            pruner = HyperbandPruner()
        else:
            pruner = MedianPruner()
        
        # Create study
        self.study = optuna.create_study(
            study_name=study_name,
            direction='maximize' if self.task_type == 'classification' else 'minimize',
            sampler=sampler,
            pruner=pruner
        )
        
        # Optimize
        self.study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=self.config.n_trials,
            timeout=self.config.timeout,
            n_jobs=self.config.n_jobs,
            show_progress_bar=True
        )
        
        search_time = time.time() - start_time
        
        # Get best model
        best_params = self.study.best_params
        self.best_model = self._create_best_model(best_params)
        
        # Create result
        result = SearchResult(
            best_params=best_params,
            best_value=self.study.best_value,
            best_model=self.best_model,
            study=self.study,
            training_history={},
            search_time=search_time,
            n_completed_trials=len([t for t in self.study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
            n_pruned_trials=len([t for t in self.study.trials if t.state == optuna.trial.TrialState.PRUNED])
        )
        
        # Print summary
        self._print_summary(result)
        
        return result
    
    def _create_best_model(self, params: Dict[str, Any]) -> nn.Module:
        """Create the best model from parameters."""
        config = {}
        n_layers = params['n_layers']
        config['n_layers'] = n_layers
        
        for i in range(n_layers):
            for key in ['layer_type', 'n_units', 'activation', 'dropout', 'n_heads']:
                param_key = f'{key}_l{i}'
                if param_key in params:
                    config[param_key] = params[param_key]
        
        return DynamicNetwork(config, self.input_dim, self.output_dim).to(self.device)
    
    def _print_summary(self, result: SearchResult):
        """Print search summary."""
        print("\n" + "="*50)
        print("NEURAL ARCHITECTURE SEARCH RESULTS")
        print("="*50)
        print(f"Total trials: {result.n_completed_trials + result.n_pruned_trials}")
        print(f"Completed trials: {result.n_completed_trials}")
        print(f"Pruned trials: {result.n_pruned_trials}")
        print(f"Search time: {result.search_time:.2f} seconds")
        print(f"\nBest value: {result.best_value:.4f}")
        print(f"\nBest architecture:")
        
        # Print architecture details
        n_layers = result.best_params['n_layers']
        print(f"  Number of layers: {n_layers}")
        
        for i in range(n_layers):
            layer_type = result.best_params.get(f'layer_type_l{i}', 'linear')
            n_units = result.best_params.get(f'n_units_l{i}')
            activation = result.best_params.get(f'activation_l{i}')
            dropout = result.best_params.get(f'dropout_l{i}')
            
            print(f"  Layer {i+1}: {layer_type}, {n_units} units, {activation}, dropout={dropout:.3f}")
        
        print(f"\nTraining hyperparameters:")
        print(f"  Learning rate: {result.best_params['learning_rate']:.5f}")
        print(f"  Batch size: {result.best_params['batch_size']}")
        print(f"  Optimizer: {result.best_params['optimizer']}")
        print(f"  Weight decay: {result.best_params['weight_decay']:.6f}")
        print("="*50)

## 4. Advanced Training Techniques

In [ ]:
class AdvancedTrainer(pl.LightningModule):
    """PyTorch Lightning module with advanced training techniques."""
    
    def __init__(self, 
                 model: nn.Module,
                 learning_rate: float = 1e-3,
                 weight_decay: float = 1e-5,
                 task_type: str = 'classification',
                 n_classes: int = 10,
                 use_mixup: bool = True,
                 use_cutmix: bool = False,
                 label_smoothing: float = 0.1,
                 use_sam: bool = False,
                 use_swa: bool = True):
        """
        Initialize advanced trainer.
        
        Parameters:
        -----------
        model : nn.Module
            Model to train
        learning_rate : float
            Learning rate
        weight_decay : float
            Weight decay for regularization
        task_type : str
            Task type ('classification', 'regression')
        n_classes : int
            Number of classes for classification
        use_mixup : bool
            Use MixUp augmentation
        use_cutmix : bool
            Use CutMix augmentation
        label_smoothing : float
            Label smoothing factor
        use_sam : bool
            Use Sharpness Aware Minimization
        use_swa : bool
            Use Stochastic Weight Averaging
        """
        super().__init__()
        self.save_hyperparameters(ignore=['model'])
        
        self.model = model
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.task_type = task_type
        self.n_classes = n_classes
        self.use_mixup = use_mixup
        self.use_cutmix = use_cutmix
        self.label_smoothing = label_smoothing
        self.use_sam = use_sam
        self.use_swa = use_swa
        
        # Loss function
        if task_type == 'classification':
            self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        else:
            self.criterion = nn.MSELoss()
        
        # Metrics
        self.train_acc = pl.metrics.Accuracy() if task_type == 'classification' else None
        self.val_acc = pl.metrics.Accuracy() if task_type == 'classification' else None
    
    def mixup_data(self, x: torch.Tensor, y: torch.Tensor, 
                   alpha: float = 1.0) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
        """Apply MixUp augmentation."""
        if alpha > 0:
            lam = np.random.beta(alpha, alpha)
        else:
            lam = 1
        
        batch_size = x.size(0)
        index = torch.randperm(batch_size).to(x.device)
        
        mixed_x = lam * x + (1 - lam) * x[index]
        y_a, y_b = y, y[index]
        
        return mixed_x, y_a, y_b, lam
    
    def cutmix_data(self, x: torch.Tensor, y: torch.Tensor, 
                    alpha: float = 1.0) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
        """Apply CutMix augmentation."""
        if alpha > 0:
            lam = np.random.beta(alpha, alpha)
        else:
            lam = 1
        
        batch_size = x.size(0)
        index = torch.randperm(batch_size).to(x.device)
        
        # For 1D data, apply CutMix on feature dimension
        if len(x.shape) == 2:  # (batch, features)
            feat_size = x.size(1)
            cut_size = int(feat_size * (1 - lam))
            cut_start = np.random.randint(0, feat_size - cut_size + 1)
            
            x[:, cut_start:cut_start+cut_size] = x[index, cut_start:cut_start+cut_size]
        
        y_a, y_b = y, y[index]
        return x, y_a, y_b, lam
    
    def training_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        """Training step."""
        x, y = batch
        
        # Apply augmentation
        if self.use_mixup and np.random.random() > 0.5:
            x, y_a, y_b, lam = self.mixup_data(x, y)
            logits = self.model(x)
            loss = lam * self.criterion(logits, y_a) + (1 - lam) * self.criterion(logits, y_b)
        elif self.use_cutmix and np.random.random() > 0.5:
            x, y_a, y_b, lam = self.cutmix_data(x, y)
            logits = self.model(x)
            loss = lam * self.criterion(logits, y_a) + (1 - lam) * self.criterion(logits, y_b)
        else:
            logits = self.model(x)
            loss = self.criterion(logits, y)
        
        # Logging
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        
        if self.task_type == 'classification' and not (self.use_mixup or self.use_cutmix):
            self.train_acc(logits, y)
            self.log('train_acc', self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        
        return loss
    
    def validation_step(self, batch: Tuple[torch.Tensor, torch.Tensor], batch_idx: int) -> torch.Tensor:
        """Validation step."""
        x, y = batch
        logits = self.model(x)
        loss = self.criterion(logits, y)
        
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        
        if self.task_type == 'classification':
            self.val_acc(logits, y)
            self.log('val_acc', self.val_acc, on_step=False, on_epoch=True, prog_bar=True)
        
        return loss
    
    def configure_optimizers(self) -> Dict:
        """Configure optimizers and schedulers."""
        # Optimizer
        if self.use_sam:
            # SAM optimizer wrapper would go here
            optimizer = optim.AdamW(self.parameters(), 
                                  lr=self.learning_rate,
                                  weight_decay=self.weight_decay)
        else:
            optimizer = optim.AdamW(self.parameters(),
                                  lr=self.learning_rate,
                                  weight_decay=self.weight_decay)
        
        # Scheduler
        scheduler = {
            'scheduler': OneCycleLR(
                optimizer,
                max_lr=self.learning_rate,
                total_steps=self.trainer.estimated_stepping_batches,
                pct_start=0.1,
                anneal_strategy='cos',
                div_factor=25,
                final_div_factor=1e4
            ),
            'interval': 'step',
            'frequency': 1
        }
        
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}
    
    def on_train_epoch_end(self):
        """Called at the end of training epoch."""
        # Log current learning rate
        current_lr = self.optimizers().param_groups[0]['lr']
        self.log('learning_rate', current_lr, on_step=False, on_epoch=True)

## 5. Visualization and Analysis

In [ ]:
class NASVisualizer:
    """Visualization tools for NAS results."""
    
    @staticmethod
    def plot_optimization_history(study: optuna.Study):
        """Plot optimization history."""
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=['Optimization History', 'Parallel Coordinate Plot',
                          'Parameter Importances', 'Trial Duration']
        )
        
        # Optimization history
        trials = study.trials
        values = [t.value for t in trials if t.value is not None]
        
        fig.add_trace(
            go.Scatter(x=list(range(len(values))), y=values,
                      mode='markers+lines', name='Trial Value'),
            row=1, col=1
        )
        
        # Best value line
        best_values = [max(values[:i+1]) if study.direction == optuna.study.StudyDirection.MAXIMIZE 
                      else min(values[:i+1]) for i in range(len(values))]
        fig.add_trace(
            go.Scatter(x=list(range(len(best_values))), y=best_values,
                      mode='lines', name='Best Value',
                      line=dict(color='red', width=2)),
            row=1, col=1
        )
        
        # Trial duration
        durations = [(t.datetime_complete - t.datetime_start).total_seconds() 
                    for t in trials if t.datetime_complete is not None]
        
        fig.add_trace(
            go.Bar(x=list(range(len(durations))), y=durations,
                  name='Duration (s)'),
            row=2, col=2
        )
        
        fig.update_layout(height=800, showlegend=True,
                         title_text="Neural Architecture Search Results")
        
        return fig
    
    @staticmethod
    def plot_param_distributions(study: optuna.Study, params: List[str] = None):
        """Plot parameter distributions."""
        if params is None:
            params = list(study.best_params.keys())[:6]  # Top 6 params
        
        n_params = len(params)
        n_cols = 3
        n_rows = (n_params + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
        axes = axes.flatten() if n_rows > 1 else [axes]
        
        for i, param in enumerate(params):
            if i >= len(axes):
                break
                
            values = [t.params.get(param) for t in study.trials 
                     if param in t.params and t.value is not None]
            
            if values:
                axes[i].hist(values, bins=20, edgecolor='black', alpha=0.7)
                axes[i].axvline(study.best_params.get(param), color='red', 
                               linestyle='--', label='Best')
                axes[i].set_xlabel(param)
                axes[i].set_ylabel('Count')
                axes[i].legend()
                axes[i].grid(True, alpha=0.3)
        
        # Hide unused axes
        for i in range(len(params), len(axes)):
            axes[i].set_visible(False)
        
        plt.suptitle('Parameter Distributions', fontsize=16)
        plt.tight_layout()
        
        return fig
    
    @staticmethod
    def create_architecture_diagram(model: nn.Module):
        """Create architecture diagram."""
        from torchinfo import summary
        
        # Get model summary
        model_summary = summary(model, input_size=(1, model.input_dim),
                               verbose=0, device='cpu')
        
        return str(model_summary)

## 6. Example Usage

In [ ]:
# Generate synthetic dataset for demonstration
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create dataset
X, y = make_classification(
    n_samples=10000,
    n_features=50,
    n_informative=30,
    n_redundant=10,
    n_classes=5,
    n_clusters_per_class=2,
    random_state=42
)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Dataset shape:")
print(f"  Training: {X_train.shape}")
print(f"  Validation: {X_val.shape}")
print(f"  Test: {X_test.shape}")
print(f"  Number of classes: {len(np.unique(y))}")

In [ ]:
# Configure NAS
nas_config = NASConfig(
    min_layers=1,
    max_layers=5,
    min_units=32,
    max_units=512,
    n_trials=20,  # Reduced for demonstration
    max_epochs=30,
    patience=5,
    sampler_type='tpe',
    pruner_type='median'
)

# Initialize NAS
nas = NeuralArchitectureSearch(
    input_dim=X_train.shape[1],
    output_dim=len(np.unique(y_train)),
    task_type='classification',
    config=nas_config
)

# Run search
result = nas.search(X_train, y_train, X_val, y_val)

In [ ]:
# Visualize results
visualizer = NASVisualizer()

# Plot optimization history
fig = visualizer.plot_optimization_history(result.study)
fig.show()

# Plot parameter distributions
plt_fig = visualizer.plot_param_distributions(
    result.study,
    params=['n_layers', 'learning_rate', 'batch_size', 'dropout_l0']
)
plt.show()

In [ ]:
# Evaluate best model on test set
best_model = result.best_model
best_model.eval()

# Convert to tensors
X_test_tensor = torch.FloatTensor(X_test).to(nas.device)
y_test_tensor = torch.LongTensor(y_test).to(nas.device)

# Make predictions
with torch.no_grad():
    outputs = best_model(X_test_tensor)
    _, predictions = torch.max(outputs, 1)
    
    # Calculate accuracy
    correct = (predictions == y_test_tensor).sum().item()
    accuracy = correct / len(y_test_tensor)

print(f"\nTest Set Performance:")
print(f"  Accuracy: {accuracy:.4f}")

# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, predictions.cpu().numpy())
print(f"\nClassification Report:")
print(classification_report(y_test, predictions.cpu().numpy()))

In [ ]:
# Train best model with advanced techniques
print("\nTraining best model with advanced techniques...\n")

# Create data loaders
train_dataset = TensorDataset(
    torch.FloatTensor(X_train),
    torch.LongTensor(y_train)
)
val_dataset = TensorDataset(
    torch.FloatTensor(X_val),
    torch.LongTensor(y_val)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Initialize advanced trainer
advanced_trainer = AdvancedTrainer(
    model=result.best_model,
    learning_rate=result.best_params['learning_rate'],
    weight_decay=result.best_params['weight_decay'],
    task_type='classification',
    n_classes=len(np.unique(y_train)),
    use_mixup=True,
    use_cutmix=False,
    label_smoothing=0.1
)

# Configure PyTorch Lightning trainer
trainer = pl.Trainer(
    max_epochs=50,
    gpus=1 if torch.cuda.is_available() else 0,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=10, mode='min'),
        ModelCheckpoint(monitor='val_acc', mode='max', save_top_k=1),
        LearningRateMonitor()
    ],
    logger=TensorBoardLogger('logs', name='advanced_training'),
    enable_progress_bar=True,
    deterministic=True
)

# Train
trainer.fit(advanced_trainer, train_loader, val_loader)

## 7. Save and Export Results

In [ ]:
# Save results
import os

# Create output directory
output_dir = Path('nas_results')
output_dir.mkdir(exist_ok=True)

# Save best model
torch.save({
    'model_state_dict': result.best_model.state_dict(),
    'config': result.best_params,
    'input_dim': nas.input_dim,
    'output_dim': nas.output_dim
}, output_dir / 'best_model.pt')

# Save study
with open(output_dir / 'study.pkl', 'wb') as f:
    pickle.dump(result.study, f)

# Save configuration
with open(output_dir / 'nas_config.json', 'w') as f:
    config_dict = {
        'min_layers': nas_config.min_layers,
        'max_layers': nas_config.max_layers,
        'min_units': nas_config.min_units,
        'max_units': nas_config.max_units,
        'n_trials': nas_config.n_trials,
        'search_time': result.search_time,
        'best_value': result.best_value
    }
    json.dump(config_dict, f, indent=2)

# Export results summary
summary = {
    'best_params': result.best_params,
    'best_value': result.best_value,
    'n_completed_trials': result.n_completed_trials,
    'n_pruned_trials': result.n_pruned_trials,
    'search_time': result.search_time,
    'test_accuracy': accuracy
}

with open(output_dir / 'results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Results saved to {output_dir}/")

## Summary

This notebook demonstrates a comprehensive Neural Architecture Search (NAS) framework with:

### Key Features:
1. **Automated Architecture Search**:
   - Dynamic network construction
   - Support for different layer types (linear, residual, attention)
   - Hyperparameter optimization

2. **Advanced Training Techniques**:
   - MixUp and CutMix augmentation
   - Label smoothing
   - One-cycle learning rate scheduling
   - Gradient clipping

3. **Search Strategies**:
   - TPE (Tree-structured Parzen Estimator)
   - Random search
   - CMA-ES
   - Pruning for efficiency

4. **Comprehensive Evaluation**:
   - Optimization history visualization
   - Parameter importance analysis
   - Performance metrics

### Use Cases:
- Automated model design for new datasets
- Hyperparameter tuning
- Architecture optimization
- Transfer learning base model selection

The framework is production-ready and can be easily extended with additional layer types, search strategies, and training techniques.